# TODO 6.7.2：Prompt 与确定性计算消融

在同一个 FP16 模型和 883 条 FinQA dev 上比较四种 Prompt 方案。few-shot 示例只从 calibration/train 读取；不要把 dev/test 答案放入 Prompt。

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import gc, json, sys
from pathlib import Path
import torch

PROJECT_DIR = Path('/content/drive/MyDrive/OptimizationSystem')
DATASET_DIR = Path('/content/drive/MyDrive/banking_llm_project/datasets')
MODEL_MANIFEST = Path('/content/drive/MyDrive/banking_llm_project/model_input_real/model_manifest.json')
BASELINE_CONFIG = PROJECT_DIR / 'configs/baseline.json'
QUALITY_CONFIG = PROJECT_DIR / 'configs/quality_repair.json'
RESULT_DIR = PROJECT_DIR / 'out/quality_repair_6_7_2'
sys.path.insert(0, str(PROJECT_DIR))
from src.baseline.service import BaselineService
from src.data.finqa_assets import load_assets
from src.evaluation.quality_repair import run_quality_repair_experiment

assert torch.cuda.is_available(), '请切换到 NVIDIA GPU。'
assets = load_assets(DATASET_DIR)
quality_config = json.loads(QUALITY_CONFIG.read_text(encoding='utf-8'))
dataset_manifest = json.loads((DATASET_DIR / 'manifest.json').read_text(encoding='utf-8'))
service = BaselineService.from_local_model(BASELINE_CONFIG, MODEL_MANIFEST, RESULT_DIR / 'quality_service.jsonl')
report = run_quality_repair_experiment(
    service,
    assets.quality_dev,
    dataset_manifest,
    quality_config,
    RESULT_DIR,
    calibration=assets.calibration,
)
print(json.dumps(report, ensure_ascii=False, indent=2))
del service
gc.collect()
torch.cuda.empty_cache()

Mounted at /content/drive


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[6.7.2] 方案 1/3 zero_shot_final_answer 开始，总进度 0.0%
[6.7.2] zero_shot_final_answer: 4/80 (5.0%), 总进度 1.7%
[6.7.2] zero_shot_final_answer: 8/80 (10.0%), 总进度 3.3%
[6.7.2] zero_shot_final_answer: 12/80 (15.0%), 总进度 5.0%
[6.7.2] zero_shot_final_answer: 16/80 (20.0%), 总进度 6.7%
[6.7.2] zero_shot_final_answer: 20/80 (25.0%), 总进度 8.3%
[6.7.2] zero_shot_final_answer: 24/80 (30.0%), 总进度 10.0%
[6.7.2] zero_shot_final_answer: 28/80 (35.0%), 总进度 11.7%
[6.7.2] zero_shot_final_answer: 32/80 (40.0%), 总进度 13.3%
[6.7.2] zero_shot_final_answer: 36/80 (45.0%), 总进度 15.0%
[6.7.2] zero_shot_final_answer: 40/80 (50.0%), 总进度 16.7%
[6.7.2] zero_shot_final_answer: 44/80 (55.0%), 总进度 18.3%
[6.7.2] zero_shot_final_answer: 48/80 (60.0%), 总进度 20.0%
[6.7.2] zero_shot_final_answer: 52/80 (65.0%), 总进度 21.7%
[6.7.2] zero_shot_final_answer: 56/80 (70.0%), 总进度 23.3%
[6.7.2] zero_shot_final_answer: 60/80 (75.0%), 总进度 25.0%
[6.7.2] zero_shot_final_answer: 64/80 (80.0%), 总进度 26.7%
[6.7.2] zero_shot_final_answer: 68/80 (85.0%),

In [3]:
import gc
import json
import subprocess
import sys
from pathlib import Path

import torch
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'huggingface_hub'])
from huggingface_hub import snapshot_download

PROJECT_DIR = Path('/content/drive/MyDrive/OptimizationSystem')
DATASET_DIR = Path('/content/drive/MyDrive/banking_llm_project/datasets')
MODEL_ROOT = Path('/content/drive/MyDrive/banking_llm_project/model_input_real')
BASELINE_CONFIG = PROJECT_DIR / 'configs/baseline.json'
QUALITY_CONFIG = PROJECT_DIR / 'configs/quality_repair.json'
RESULT_ROOT = PROJECT_DIR / 'out/quality_repair_6_7_3_model_scale'
sys.path.insert(0, str(PROJECT_DIR))

MODEL_DOWNLOADS = {
    'qwen2.5-1.5b': ('Qwen/Qwen2.5-1.5B-Instruct', 'qwen2.5-1.5b-instruct'),
    'qwen2.5-3b': ('Qwen/Qwen2.5-3B-Instruct', 'qwen2.5-3b-instruct'),
}
def model_download_complete(model_dir):
    has_config = (model_dir / 'config.json').exists()
    has_weights = (model_dir / 'model.safetensors').exists() or (model_dir / 'model.safetensors.index.json').exists()
    return has_config and has_weights

for model_key, (repo_id, directory_name) in MODEL_DOWNLOADS.items():
    target_dir = MODEL_ROOT / directory_name
    if model_download_complete(target_dir):
        print(f'{model_key}: 已存在，跳过下载 -> {target_dir}')
    else:
        print(f'{model_key}: 开始下载 {repo_id} -> {target_dir}')
        target_dir.mkdir(parents=True, exist_ok=True)
        snapshot_download(repo_id=repo_id, local_dir=str(target_dir))
        print(f'{model_key}: 下载完成')

from src.baseline.service import BaselineService
from src.data.finqa_assets import load_assets
from src.evaluation.quality_repair import run_quality_repair_experiment

assert torch.cuda.is_available(), '请切换到 NVIDIA GPU。'

MODEL_CANDIDATES = {
    'qwen2.5-1.5b': [
        MODEL_ROOT / 'qwen2.5-1.5b-instruct',
        MODEL_ROOT / 'Qwen2.5-1.5B-Instruct',
    ],
    'qwen2.5-3b': [
        MODEL_ROOT / 'qwen2.5-3b-instruct',
        MODEL_ROOT / 'Qwen2.5-3B-Instruct',
    ],
}
MODEL_IDS = {
    'qwen2.5-1.5b': 'Qwen/Qwen2.5-1.5B-Instruct',
    'qwen2.5-3b': 'Qwen/Qwen2.5-3B-Instruct',
}

def resolve_model_dir(candidates):
    for candidate in candidates:
        if candidate.is_dir():
            return candidate
    expected = '\n'.join(str(candidate) for candidate in candidates)
    raise FileNotFoundError(f'找不到模型目录，请检查以下路径：\n{expected}')

def write_local_manifest(model_key, model_dir, output_dir):
    model_id = MODEL_IDS[model_key]
    manifest = {
        'schema_version': 1,
        'model_id': model_id,
        'model_path': str(model_dir),
        'version': model_id,
        'commit': 'colab-local',
        'weight_files': [],
        'config_file': 'config.json',
        'tokenizer_files': [],
        'expected_sha256': {},
        'runtime': {
            'framework': 'transformers',
            'model_class': 'causal_lm',
            'dtype': 'float16',
            'device': 'cuda:0',
            'trust_remote_code': False,
        },
    }
    manifest_path = output_dir / 'model_manifest.json'
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    return manifest_path

assets = load_assets(DATASET_DIR)
quality_config = json.loads(QUALITY_CONFIG.read_text(encoding='utf-8'))
dataset_manifest = json.loads((DATASET_DIR / 'manifest.json').read_text(encoding='utf-8'))
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

all_reports = {}
for model_key, candidates in MODEL_CANDIDATES.items():
    model_dir = resolve_model_dir(candidates)
    model_result_dir = RESULT_ROOT / model_key
    model_result_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = write_local_manifest(model_key, model_dir, model_result_dir)
    print(f'\n===== {MODEL_IDS[model_key]} =====')
    print(f'模型目录: {model_dir}')
    print(f'结果目录: {model_result_dir}')
    service = BaselineService.from_local_model(
        BASELINE_CONFIG,
        manifest_path,
        model_result_dir / 'quality_service.jsonl',
    )
    report = run_quality_repair_experiment(
        service,
        assets.quality_dev,
        dataset_manifest,
        quality_config,
        model_result_dir,
        calibration=assets.calibration,
    )
    report['model'] = {
        'key': model_key,
        'model_id': MODEL_IDS[model_key],
        'model_path': str(model_dir),
    }
    (model_result_dir / 'comparison_report.json').write_text(
        json.dumps(report, ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8',
    )
    all_reports[model_key] = report
    for variant in report['variants']:
        metrics = variant['metrics']
        print(
            f"{variant['variant']}: parse={metrics['parse_rate']:.2%}, "
            f"accuracy={metrics['numeric_accuracy']:.2%}, "
            f"correct={metrics['correct']}/{metrics['total']}"
        )
    del service
    gc.collect()
    torch.cuda.empty_cache()

print('\n===== 模型规模对比完成 =====')
for model_key, report in all_reports.items():
    print(model_key, '->', RESULT_ROOT / model_key / 'comparison_report.json')

qwen2.5-1.5b: 开始下载 Qwen/Qwen2.5-1.5B-Instruct -> /content/drive/MyDrive/banking_llm_project/model_input_real/qwen2.5-1.5b-instruct


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

qwen2.5-1.5b: 下载完成
qwen2.5-3b: 开始下载 Qwen/Qwen2.5-3B-Instruct -> /content/drive/MyDrive/banking_llm_project/model_input_real/qwen2.5-3b-instruct


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

...

qwen2.5-3b: 下载完成

===== Qwen/Qwen2.5-1.5B-Instruct =====
模型目录: /content/drive/MyDrive/banking_llm_project/model_input_real/qwen2.5-1.5b-instruct
结果目录: /content/drive/MyDrive/OptimizationSystem/out/quality_repair_6_7_3_model_scale/qwen2.5-1.5b


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[6.7.2] 方案 1/3 zero_shot_final_answer 开始，总进度 0.0%
[6.7.2] zero_shot_final_answer: 4/80 (5.0%), 总进度 1.7%
[6.7.2] zero_shot_final_answer: 8/80 (10.0%), 总进度 3.3%
[6.7.2] zero_shot_final_answer: 12/80 (15.0%), 总进度 5.0%
[6.7.2] zero_shot_final_answer: 16/80 (20.0%), 总进度 6.7%
[6.7.2] zero_shot_final_answer: 20/80 (25.0%), 总进度 8.3%
[6.7.2] zero_shot_final_answer: 24/80 (30.0%), 总进度 10.0%
[6.7.2] zero_shot_final_answer: 28/80 (35.0%), 总进度 11.7%
[6.7.2] zero_shot_final_answer: 32/80 (40.0%), 总进度 13.3%
[6.7.2] zero_shot_final_answer: 36/80 (45.0%), 总进度 15.0%
[6.7.2] zero_shot_final_answer: 40/80 (50.0%), 总进度 16.7%
[6.7.2] zero_shot_final_answer: 44/80 (55.0%), 总进度 18.3%
[6.7.2] zero_shot_final_answer: 48/80 (60.0%), 总进度 20.0%
[6.7.2] zero_shot_final_answer: 52/80 (65.0%), 总进度 21.7%
[6.7.2] zero_shot_final_answer: 56/80 (70.0%), 总进度 23.3%
[6.7.2] zero_shot_final_answer: 60/80 (75.0%), 总进度 25.0%
[6.7.2] zero_shot_final_answer: 64/80 (80.0%), 总进度 26.7%
[6.7.2] zero_shot_final_answer: 68/80 (85.0%),

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

[6.7.2] 方案 1/3 zero_shot_final_answer 开始，总进度 0.0%
[6.7.2] zero_shot_final_answer: 4/80 (5.0%), 总进度 1.7%
[6.7.2] zero_shot_final_answer: 8/80 (10.0%), 总进度 3.3%
[6.7.2] zero_shot_final_answer: 12/80 (15.0%), 总进度 5.0%
[6.7.2] zero_shot_final_answer: 16/80 (20.0%), 总进度 6.7%
[6.7.2] zero_shot_final_answer: 20/80 (25.0%), 总进度 8.3%
[6.7.2] zero_shot_final_answer: 24/80 (30.0%), 总进度 10.0%
[6.7.2] zero_shot_final_answer: 28/80 (35.0%), 总进度 11.7%
[6.7.2] zero_shot_final_answer: 32/80 (40.0%), 总进度 13.3%
[6.7.2] zero_shot_final_answer: 36/80 (45.0%), 总进度 15.0%
[6.7.2] zero_shot_final_answer: 40/80 (50.0%), 总进度 16.7%
[6.7.2] zero_shot_final_answer: 44/80 (55.0%), 总进度 18.3%
[6.7.2] zero_shot_final_answer: 48/80 (60.0%), 总进度 20.0%
[6.7.2] zero_shot_final_answer: 52/80 (65.0%), 总进度 21.7%
[6.7.2] zero_shot_final_answer: 56/80 (70.0%), 总进度 23.3%
[6.7.2] zero_shot_final_answer: 60/80 (75.0%), 总进度 25.0%
[6.7.2] zero_shot_final_answer: 64/80 (80.0%), 总进度 26.7%
[6.7.2] zero_shot_final_answer: 68/80 (85.0%),